In [1]:
"""
NASA C-MAPSS Jet Engine Dataset - Sliding Window Preprocessing
Converts raw time-series data into 3D tensors for sequence modeling.
 
Output tensor shapes:
  X_train: (N_samples, window_size, n_features)  -> 3D input tensor
  y_train: (N_samples,)                          -> RUL labels
  X_test:  (N_engines, window_size, n_features)  -> last window per engine
  y_test:  (N_engines,)                          -> true RUL labels
"""
 
import numpy as np
import pandas as pd
from pathlib import Path
import os

c:\Users\gtanu\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


# ──────────────────────────────────────────────
# Configuration
# ──────────────────────────────────────────────

In [10]:
DATA_DIR   = Path("C:\\Users\\gtanu\\Downloads\\IIoT-PM\\CMAPSS Jet Engine Simulated Data")
OUTPUT_DIR = Path("C:\\Users\\gtanu\\Downloads\\IIoT-PM\\CMAPSS Jet Engine Simulated Data\\processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
 
WINDOW_SIZE  = 30          # sequence length (cycles)
MAX_RUL      = 125         # piece-wise linear RUL cap (common in literature)
DATASETS     = ["FD001", "FD002", "FD003", "FD004"]
 
COLUMNS = (
    ["unit", "cycle"]
    + [f"op_{i}" for i in range(1, 4)]
    + [f"s{i}"   for i in range(1, 22)]
)
 
# Sensors with near-zero variance (constant across all conditions) – drop these
CONSTANT_SENSORS = ["s1", "s5", "s10", "s16", "s18", "s19"]
# Operational settings (kept as features)
OP_SETTINGS = ["op_1", "op_2", "op_3"]

# ──────────────────────────────────────────────
# Helper functions
# ──────────────────────────────────────────────

In [11]:
def load_raw(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep=r"\s+", header=None, names=COLUMNS)
    return df

In [12]:
def add_rul_column(df: pd.DataFrame, max_rul: int = MAX_RUL) -> pd.DataFrame:
    """Compute piece-wise linear (capped) RUL for training data."""
    rul_per_unit = (
        df.groupby("unit")["cycle"]
        .max()
        .reset_index()
        .rename(columns={"cycle": "max_cycle"})
    )
    df = df.merge(rul_per_unit, on="unit")
    df["RUL"] = df["max_cycle"] - df["cycle"]
    # Piece-wise linear cap
    df["RUL"] = df["RUL"].clip(upper=max_rul)
    df.drop(columns=["max_cycle"], inplace=True)
    return df

In [13]:
def normalize(train_df: pd.DataFrame,
              test_df:  pd.DataFrame,
              feature_cols: list[str]):
    """Min-max normalize using training set statistics."""
    mins  = train_df[feature_cols].min()
    maxs  = train_df[feature_cols].max()
    rng   = (maxs - mins).replace(0, 1)          # avoid /0 on constant cols
 
    train_df = train_df.copy()
    test_df  = test_df.copy()
    train_df[feature_cols] = (train_df[feature_cols] - mins) / rng
    test_df[feature_cols]  = (test_df[feature_cols]  - mins) / rng
    return train_df, test_df, mins, maxs

In [15]:
def sliding_windows_train(df: pd.DataFrame,
                          feature_cols: list[str],
                          window: int) -> tuple[np.ndarray, np.ndarray]:
    """
    For each engine in the training set, slide a window across ALL cycles.
    Yields one sample per valid (engine, end-cycle) pair.
 
    Returns
    -------
    X : np.ndarray  shape (N, window, n_features)
    y : np.ndarray  shape (N,)
    """
    X_list, y_list = [], []
    for _, grp in df.groupby("unit"):
        grp = grp.sort_values("cycle")
        data = grp[feature_cols].values    # (T, F)
        rul  = grp["RUL"].values           # (T,)
        T    = len(data)
        if T < window:
            continue                       # engine too short – skip
        for end in range(window, T + 1):
            X_list.append(data[end - window : end])   # (window, F)
            y_list.append(rul[end - 1])               # RUL at last step
    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32)

In [16]:
def sliding_windows_test(df: pd.DataFrame,
                         feature_cols: list[str],
                         window: int) -> np.ndarray:
    """
    For each engine in the test set, extract ONE window: the last `window` cycles.
    Pads with zeros at the front if the engine has fewer than `window` cycles.
 
    Returns
    -------
    X : np.ndarray  shape (N_engines, window, n_features)
    """
    X_list = []
    for _, grp in df.groupby("unit"):
        grp  = grp.sort_values("cycle")
        data = grp[feature_cols].values    # (T, F)
        T    = len(data)
        if T >= window:
            X_list.append(data[-window:])  # last `window` rows
        else:
            pad = np.zeros((window - T, data.shape[1]), dtype=np.float32)
            X_list.append(np.vstack([pad, data]))
    return np.array(X_list, dtype=np.float32)

In [17]:
def process_dataset(name: str):
    print(f"\n{'='*55}")
    print(f"  Processing {name}")
    print(f"{'='*55}")
 
    # ── Load ──────────────────────────────────────────────
    train_raw = load_raw(DATA_DIR / f"train_{name}.txt")
    test_raw  = load_raw(DATA_DIR / f"test_{name}.txt")
    rul_true  = np.loadtxt(DATA_DIR / f"RUL_{name}.txt", dtype=np.float32)
 
    print(f"  Train rows : {len(train_raw):,}  |  engines: {train_raw['unit'].nunique()}")
    print(f"  Test rows  : {len(test_raw):,}   |  engines: {test_raw['unit'].nunique()}")
 
    # ── Feature selection ─────────────────────────────────
    sensor_cols  = [c for c in COLUMNS if c.startswith("s") and c not in CONSTANT_SENSORS]
    feature_cols = OP_SETTINGS + sensor_cols
    print(f"  Features   : {len(feature_cols)}  {feature_cols}")
 
    # ── RUL label for training ────────────────────────────
    train_raw = add_rul_column(train_raw, max_rul=MAX_RUL)
 
    # ── Normalize ─────────────────────────────────────────
    train_norm, test_norm, mins, maxs = normalize(train_raw, test_raw, feature_cols)
 
    # ── Sliding windows ───────────────────────────────────
    X_train, y_train = sliding_windows_train(train_norm, feature_cols, WINDOW_SIZE)
    X_test            = sliding_windows_test (test_norm,  feature_cols, WINDOW_SIZE)
    y_test            = rul_true.clip(max=MAX_RUL)   # cap ground-truth too
 
    print(f"\n  ┌─ Output tensor shapes ─────────────────────────┐")
    print(f"  │  X_train : {X_train.shape}  (samples × window × features)")
    print(f"  │  y_train : {y_train.shape}")
    print(f"  │  X_test  : {X_test.shape}   (engines × window × features)")
    print(f"  │  y_test  : {y_test.shape}")
    print(f"  └────────────────────────────────────────────────┘")
    print(f"\n  y_train  min={y_train.min():.1f}  max={y_train.max():.1f}  mean={y_train.mean():.1f}")
    print(f"  y_test   min={y_test.min():.1f}  max={y_test.max():.1f}  mean={y_test.mean():.1f}")
 
    # ── Save ──────────────────────────────────────────────
    out_path = OUTPUT_DIR / f"{name}_tensors.npz"
    np.savez_compressed(
        out_path,
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
        feature_names=np.array(feature_cols),
        window_size=np.array([WINDOW_SIZE]),
        max_rul=np.array([MAX_RUL]),
    )
    print(f"\n  Saved → {out_path.name}  ({out_path.stat().st_size / 1e6:.2f} MB)")
 
    return {
        "dataset":    name,
        "n_features": len(feature_cols),
        "X_train":    X_train.shape,
        "y_train":    y_train.shape,
        "X_test":     X_test.shape,
        "y_test":     y_test.shape,
    }

# ──────────────────────────────────────────────
# Main
# ──────────────────────────────────────────────

In [18]:
if __name__ == "__main__":
    results = []
    for ds in DATASETS:
        results.append(process_dataset(ds))
 
    print(f"\n\n{'='*55}")
    print("  SUMMARY – All Datasets")
    print(f"{'='*55}")
    print(f"{'Dataset':<8} {'X_train shape':<30} {'X_test shape':<25} {'Features'}")
    print("-"*80)
    for r in results:
        print(f"{r['dataset']:<8} {str(r['X_train']):<30} {str(r['X_test']):<25} {r['n_features']}")
    print(f"\nWindow size : {WINDOW_SIZE} cycles")
    print(f"RUL cap     : {MAX_RUL} cycles")
    print(f"\nAll .npz files saved to: {OUTPUT_DIR}")
    print("\nUsage example:")
    print("  import numpy as np")
    print("  d = np.load('FD001_tensors.npz')")
    print("  X_train, y_train = d['X_train'], d['y_train']")
    print("  X_test,  y_test  = d['X_test'],  d['y_test']")
    print("  # X_train.shape → (N_samples, 30, 17) for FD001")


  Processing FD001
  Train rows : 20,631  |  engines: 100
  Test rows  : 13,096   |  engines: 100
  Features   : 18  ['op_1', 'op_2', 'op_3', 's2', 's3', 's4', 's6', 's7', 's8', 's9', 's11', 's12', 's13', 's14', 's15', 's17', 's20', 's21']

  ┌─ Output tensor shapes ─────────────────────────┐
  │  X_train : (17731, 30, 18)  (samples × window × features)
  │  y_train : (17731,)
  │  X_test  : (100, 30, 18)   (engines × window × features)
  │  y_test  : (100,)
  └────────────────────────────────────────────────┘

  y_train  min=0.0  max=125.0  mean=80.6
  y_test   min=7.0  max=125.0  mean=74.4

  Saved → FD001_tensors.npz  (1.41 MB)

  Processing FD002
  Train rows : 53,759  |  engines: 260
  Test rows  : 33,991   |  engines: 259
  Features   : 18  ['op_1', 'op_2', 'op_3', 's2', 's3', 's4', 's6', 's7', 's8', 's9', 's11', 's12', 's13', 's14', 's15', 's17', 's20', 's21']

  ┌─ Output tensor shapes ─────────────────────────┐
  │  X_train : (46219, 30, 18)  (samples × window × features)
  │